In [ ]:
# ---------------- Imports ----------------
import os
import json
from collections import Counter
from collections import defaultdict

import yaml
import pandas as pd
from transformers import AutoTokenizer
import numpy as np



In [ ]:
# ---------------- Args ----------------
DATASET_CHOICE = "20260115T095923-combined-claims-full"


model_choice = "meta-llama/Llama-3.2-3B-Instruct"



In [ ]:
# ---------------- Config ----------------

with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
DATASET_FOLDER = os.path.join(PROJ_STORE, "data", "augmented-processed")
MODELS_FOLDER = config["paths"]["models"]


TARGET_FOLDER = os.path.join(DATASET_FOLDER, DATASET_CHOICE)

# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "dataset-stats", DATASET_CHOICE)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# MODEL
model_path = os.path.join(MODELS_FOLDER, model_choice)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
#tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# -------------------------
# Workspace
# -------------------------


# Global counts
framing_counts = Counter()

# Per-subfolder counts
folder_counts = defaultdict(Counter)

# Token stats
token_lengths = []
folder_token_lengths = defaultdict(list)

# Token stats by framing type
framing_token_lengths = defaultdict(list)

# Chat-template token stats
chat_token_lengths = []
folder_chat_token_lengths = defaultdict(list)



# Main Scan
for dirpath, _, filenames in os.walk(TARGET_FOLDER):

    # Relative path for folder key
    rel_path = os.path.relpath(dirpath, TARGET_FOLDER)

    for filename in filenames:

        if not filename.endswith(".jsonl"):
            continue

        filepath = os.path.join(dirpath, filename)

        with open(filepath, "r", encoding="utf-8") as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                obj = json.loads(line)

                # ---- Framing counts ----
                framing = obj.get("framing_type")

                if framing is not None:
                    framing_counts[framing] += 1
                    folder_counts[rel_path][framing] += 1

                # ---- Token stats ----
                claim = obj.get("restated_claim")

                if claim:

                    tokens = tokenizer.encode(
                        claim, add_special_tokens=False
                    )

                    n_tokens = len(tokens)

                    token_lengths.append(n_tokens)
                    folder_token_lengths[rel_path].append(n_tokens)

                    # By framing type
                    if framing is not None:
                        framing_token_lengths[framing].append(n_tokens)


                # ---- Chat template token stats ----
                messages = obj.get("messages")

                if messages:

                    chat_text = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=False 
                    )

                    chat_tokens = tokenizer.encode(
                        chat_text,
                        add_special_tokens=False
                    )

                    n_chat_tokens = len(chat_tokens)

                    chat_token_lengths.append(n_chat_tokens)
                    folder_chat_token_lengths[rel_path].append(n_chat_tokens)

In [ ]:
# Global table

df_global = (
    pd.DataFrame(
        framing_counts.items(),
        columns=["framing_type", "count"]
    )
    .sort_values("framing_type")
    .reset_index(drop=True)
)

# Add total
total_global = df_global["count"].sum()

df_global = pd.concat(
    [
        df_global,
        pd.DataFrame([["Total", total_global]],
                     columns=["framing_type", "count"])
    ],
    ignore_index=True
)

display(df_global)

# Save

output_path = os.path.join(OUTPUT_DIR, f"{DATASET_CHOICE}-all.csv")

# Save to CSV
df_global.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

In [ ]:
# Global Token Stats

token_array = np.array(token_lengths)

global_token_stats = {
    "count": len(token_array),
    "min": int(token_array.min()),
    "max": int(token_array.max()),
    "mean": float(token_array.mean()),
    "median": float(np.median(token_array)),
    "std": float(token_array.std())
}

print("\n=== Global Token Stats (restated_claim) ===")

for k, v in global_token_stats.items():
    if isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")

# Save
df_tokens = pd.DataFrame([global_token_stats])

token_path = os.path.join(
    OUTPUT_DIR,
    f"{DATASET_CHOICE}-token-stats.csv"
)

df_tokens.to_csv(token_path, index=False)

print(f"Saved: {token_path}")



In [ ]:
# Token Stats by Framing Type

print("\n=== Token Stats by Framing Type (restated_claim) ===")

framing_token_rows = []

# Add overall first
all_arr = np.array(token_lengths)

all_stats = {
    "framing_type": "ALL",
    "count": len(all_arr),
    "min": int(all_arr.min()),
    "max": int(all_arr.max()),
    "mean": float(all_arr.mean()),
    "median": float(np.median(all_arr)),
    "std": float(all_arr.std())
}

framing_token_rows.append(all_stats)

for framing, lengths in sorted(framing_token_lengths.items()):

    arr = np.array(lengths)

    stats = {
        "framing_type": framing,
        "count": len(arr),
        "min": int(arr.min()),
        "max": int(arr.max()),
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "std": float(arr.std())
    }

    framing_token_rows.append(stats)




# Save
df_framing_tokens = pd.DataFrame(framing_token_rows)

framing_token_path = os.path.join(
    OUTPUT_DIR,
    f"{DATASET_CHOICE}-token-stats-by-framing.csv"
)

display(df_framing_tokens)
df_framing_tokens.to_csv(framing_token_path, index=False)

print(f"\nSaved: {framing_token_path}")


In [ ]:
# Global Chat-Template Token Stats

chat_array = np.array(chat_token_lengths)

global_chat_stats = {
    "count": len(chat_array),
    "min": int(chat_array.min()),
    "max": int(chat_array.max()),
    "mean": float(chat_array.mean()),
    "median": float(np.median(chat_array)),
    "std": float(chat_array.std())
}

print("\n=== Global Token Stats (chat template + messages) ===")

for k, v in global_chat_stats.items():
    if isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")

# Save
df_chat = pd.DataFrame([global_chat_stats])

chat_path = os.path.join(
    OUTPUT_DIR,
    f"{DATASET_CHOICE}-chat-token-stats.csv"
)

df_chat.to_csv(chat_path, index=False)

print(f"Saved: {chat_path}")


In [ ]:
# Per-folder tables


for folder, counter in sorted(folder_counts.items()):

    df_folder = (
        pd.DataFrame(
            counter.items(),
            columns=["framing_type", "count"]
        )
        .sort_values("framing_type")
        .reset_index(drop=True)
    )

    # Add total
    total_folder = df_folder["count"].sum()

    df_folder = pd.concat(
        [
            df_folder,
            pd.DataFrame([["Total", total_folder]],
                         columns=["framing_type", "count"])
        ],
        ignore_index=True
    )

    print(f"\n=== Folder: {folder} ===")
    display(df_folder)
    
    
    # Save
    output_path = os.path.join(OUTPUT_DIR, f"{DATASET_CHOICE}-{folder}.csv")

    # Save to CSV
    df_folder.to_csv(output_path, index=False)

    print(f"Saved to: {output_path}")
        


In [ ]:
# Per-Folder Token Stats

print("\n=== Per-Folder Token Stats ===")

rows = []

for folder, lengths in sorted(folder_token_lengths.items()):

    arr = np.array(lengths)

    rows.append({
        "folder": folder,
        "count": len(arr),
        "min": int(arr.min()),
        "max": int(arr.max()),
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "std": float(arr.std())
    })


df_folder_tokens = pd.DataFrame(rows)

display(df_folder_tokens)

# Save
folder_token_path = os.path.join(
    OUTPUT_DIR,
    f"{DATASET_CHOICE}-folder-token-stats.csv"
)

df_folder_tokens.to_csv(folder_token_path, index=False)

print(f"Saved: {folder_token_path}")

In [ ]:
# Per-Folder Chat Token Stats


print("\n=== Per-Folder Chat Token Stats ===")

rows = []

for folder, lengths in sorted(folder_chat_token_lengths.items()):

    arr = np.array(lengths)

    rows.append({
        "folder": folder,
        "count": len(arr),
        "min": int(arr.min()),
        "max": int(arr.max()),
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "std": float(arr.std())
    })

df_folder_chat = pd.DataFrame(rows)

display(df_folder_chat)

# Save
folder_chat_path = os.path.join(
    OUTPUT_DIR,
    f"{DATASET_CHOICE}-folder-chat-token-stats.csv"
)

df_folder_chat.to_csv(folder_chat_path, index=False)

print(f"Saved: {folder_chat_path}")

